In [16]:
import torch
import math
from omegaconf import OmegaConf
import torch.nn as nn
import torch.nn.functional as F


class SimilarityModule(nn.Module):
    """
    Multi-head similarity module with:
    - optional L2 normalization
    - multi-head splitting
    - learnable linear projections per head
    - masking support
    - positive/negative weighting
    - top-k similarity selection
    - aggregation: sum / softmax / logsumexp
    - temperature scaling
    - energy interpretation
    """

    def __init__(self, cfg: OmegaConf, input_dim: int = None):
        super().__init__()
        self.cfg = cfg.model.similarityModule
        self.num_heads = getattr(self.cfg, "numHeads", 1)
        self.temperature = getattr(self.cfg, "temperature", 1.0)
        self.aggregation = getattr(self.cfg, "aggregation", "sum")  # sum, softmax, logsumexp
        self.pos_weight = getattr(self.cfg, "posWeight", 1.0)
        self.neg_weight = getattr(self.cfg, "negWeight", 1.0)
        self.topk = getattr(self.cfg, "topk", None)

        # Optional learnable linear projections for multi-head
        if input_dim is not None:
            assert input_dim % self.num_heads == 0, "input_dim must be divisible by num_heads"
            d_head = input_dim // self.num_heads
            self.query_proj = nn.Linear(input_dim, input_dim, bias=False)
            self.support_proj = nn.Linear(input_dim, input_dim, bias=False)
        else:
            self.query_proj = None
            self.support_proj = None

    def forward(
        self,
        query_embedding: torch.Tensor,
        support_set_embeddings: torch.Tensor,
        padding_mask: torch.Tensor,
        support_set_size: torch.Tensor = None,
    ) -> torch.Tensor:

        B, _, D = query_embedding.shape
        _, N, _ = support_set_embeddings.shape
        d_head = D // self.num_heads

        # -------------------------------
        # Assertions
        # -------------------------------
        assert query_embedding.dim() == 3 and query_embedding.shape[1] == 1
        assert support_set_embeddings.dim() == 3
        assert padding_mask.shape == (B, N) and padding_mask.dtype == torch.bool
        if support_set_size is not None:
            valid_counts = padding_mask.sum(dim=1)
            assert torch.all(valid_counts == support_set_size), \
                "support_set_size does not match padding_mask"

        # -------------------------------
        # Optional L2 normalization
        # -------------------------------
        if getattr(self.cfg, "l2Norm", False):
            query_embedding = F.normalize(query_embedding, dim=-1, eps=1e-8)
            support_set_embeddings = F.normalize(support_set_embeddings, dim=-1, eps=1e-8)

        # -------------------------------
        # Optional learnable projections
        # -------------------------------
        if self.query_proj is not None and self.support_proj is not None:
            query_embedding = self.query_proj(query_embedding)
            support_set_embeddings = self.support_proj(support_set_embeddings)

        # -------------------------------
        # Split into multi-heads
        # -------------------------------
        query = query_embedding.view(B, 1, self.num_heads, d_head).transpose(1, 2)   # [B,H,1,d_head]
        support = support_set_embeddings.view(B, N, self.num_heads, d_head).transpose(1, 2)  # [B,H,N,d_head]

        # -------------------------------
        # Compute similarity as negative energy
        # -------------------------------
        similarities = torch.matmul(query, support.transpose(-2, -1)) / math.sqrt(d_head)
        similarities = similarities * self.temperature
        similarities = torch.nan_to_num(similarities)

        # -------------------------------
        # Masking (safe for autograd)
        # -------------------------------
        mask = padding_mask.unsqueeze(1).unsqueeze(2)
        similarities = similarities.masked_fill(~mask, float("-inf")).clone()

        # -------------------------------
        # Separate positive / negative importance
        # -------------------------------
        sim_pos = torch.clamp(similarities, min=0.0) * self.pos_weight
        sim_neg = torch.clamp(similarities, max=0.0) * self.neg_weight
        similarities = sim_pos + sim_neg

        # -------------------------------
        # Optional Top-k similarity
        # -------------------------------
        if self.topk is not None and self.topk > 0:
            k = min(self.topk, N)
            similarities, _ = torch.topk(similarities, k=k, dim=-1)

        # -------------------------------
        # Interpret similarity as energy
        # -------------------------------
        energy = -similarities  # higher similarity → lower energy

        # -------------------------------
        # Aggregation over support set
        # -------------------------------
        if self.aggregation == "sum":
            similarity_sums = torch.nan_to_num(-energy, neginf=0.0).sum(dim=-1)
        elif self.aggregation == "softmax":
            attn = torch.softmax(-energy, dim=-1)
            similarity_sums = (attn * -energy).sum(dim=-1)
        elif self.aggregation == "logsumexp":
            similarity_sums = torch.logsumexp(-energy, dim=-1)
        else:
            raise ValueError(f"Unknown aggregation: {self.aggregation}")

        # -------------------------------
        # Aggregate over heads
        # -------------------------------
        similarity_sums = similarity_sums.mean(dim=1)  # [B,1]

        # -------------------------------
        # Optional scaling for sum
        # -------------------------------
        if self.aggregation == "sum" and support_set_size is not None:
            stabilizer = 1e-8
            N = support_set_size.reshape(-1, 1).float()
            if getattr(self.cfg, "scaling", "1/N") == "1/N":
                similarity_sums = similarity_sums / (2.0 * N + stabilizer)
            elif self.cfg.scaling == "1/sqrt(N)":
                similarity_sums = similarity_sums / (2.0 * torch.sqrt(N) + stabilizer)

        return similarity_sums

    # Double-check mask semantics
    # Add assertion checks
    # Add temperature scaling
    # Add optional softmax weighting (attention version)
    # Separate positive / negative importance
    # Replace sum with log-sum-exp
    # Top-k similarity
    # Interpret similarity as energy
    # multi-head similarity
    # Learnable projections for multi-head

### Test

In [18]:
import torch
from omegaconf import OmegaConf

# from your_module import SimilarityModule

def get_cfg(
    l2=True,
    scaling="1/N",
    numHeads=2,
    temperature=1.0,
    aggregation="sum",
    posWeight=1.0,
    negWeight=1.0,
    topk=None,
):
    return OmegaConf.create({
        "model": {
            "similarityModule": {
                "l2Norm": l2,
                "scaling": scaling,
                "numHeads": numHeads,
                "temperature": temperature,
                "aggregation": aggregation,
                "posWeight": posWeight,
                "negWeight": negWeight,
                "topk": topk
            }
        }
    })


def test_shapes_and_heads():
    print("\n[TEST] Shapes & Heads")
    B, N, D = 4, 6, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    cfg = get_cfg(numHeads=4)
    sim_module = SimilarityModule(cfg)
    out = sim_module(query, support, mask, size)

    print("Output shape:", out.shape)
    assert out.shape == (B, 1)
    print("✅ Passed")


def test_aggregation_methods():
    print("\n[TEST] Aggregation Methods")
    B, N, D = 2, 5, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    for agg in ["sum", "softmax", "logsumexp"]:
        cfg = get_cfg(aggregation=agg)
        sim_module = SimilarityModule(cfg)
        out = sim_module(query, support, mask, size)
        print(f"{agg}: {out.squeeze().tolist()}")
    print("✅ Passed")


def test_topk():
    print("\n[TEST] Top-K Selection")
    B, N, D = 2, 6, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    for k in [None, 3, 5]:
        cfg = get_cfg(topk=k)
        sim_module = SimilarityModule(cfg)
        out = sim_module(query, support, mask, size)
        print(f"Top-K={k}: {out.squeeze().tolist()}")
    print("✅ Passed")


def test_temperature_scaling():
    print("\n[TEST] Temperature Scaling")
    B, N, D = 2, 5, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    for temp in [0.5, 1.0, 2.0]:
        cfg = get_cfg(temperature=temp)
        sim_module = SimilarityModule(cfg)
        out = sim_module(query, support, mask, size)
        print(f"Temp={temp}: {out.squeeze().tolist()}")
    print("✅ Passed")


def test_pos_neg_weighting():
    print("\n[TEST] Pos/Neg Weighting")
    B, N, D = 2, 4, 8
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    for pw, nw in [(1.0, 1.0), (2.0, 0.5), (0.5, 2.0)]:
        cfg = get_cfg(posWeight=pw, negWeight=nw)
        sim_module = SimilarityModule(cfg)
        out = sim_module(query, support, mask, size)
        print(f"pos={pw}, neg={nw}: {out.squeeze().tolist()}")
    print("✅ Passed")


def test_variable_support_size_and_masking():
    print("\n[TEST] Variable Support Size + Masking")
    B, N, D = 3, 6, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.tensor([
        [1,1,1,0,0,0],
        [1,1,1,1,0,0],
        [1,1,1,1,1,1],
    ]).bool()
    size = torch.tensor([3, 4, 6])

    cfg = get_cfg()
    sim_module = SimilarityModule(cfg)
    out = sim_module(query, support, mask, size)
    print("Output:", out)
    assert torch.isfinite(out).all()
    print("✅ Passed")


def test_gradient_flow():
    print("\n[TEST] Gradient Flow")
    B, N, D = 2, 5, 16
    query = torch.randn(B, 1, D, requires_grad=True)
    support = torch.randn(B, N, D, requires_grad=True)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    cfg = get_cfg()
    sim_module = SimilarityModule(cfg)
    out = sim_module(query, support, mask, size)
    loss = out.mean()
    loss.backward()

    print("Query grad mean:", query.grad.abs().mean().item())
    print("Support grad mean:", support.grad.abs().mean().item())
    assert query.grad is not None
    assert support.grad is not None
    print("✅ Passed")


def test_empty_support():
    print("\n[TEST] Empty Support")
    B, N, D = 2, 0, 16
    query = torch.randn(B, 1, D)
    support = torch.empty(B, N, D)
    mask = torch.zeros(B, N).bool()
    size = torch.zeros(B, dtype=torch.long)
    cfg = get_cfg()
    sim_module = SimilarityModule(cfg, input_dim=D)
    out = sim_module(query, support, mask, size)
    print("Output:", out)
    assert torch.isfinite(out).all()
    print("✅ Passed")


def test_single_support_vector():
    print("\n[TEST] Single Support Vector")
    B, N, D = 2, 1, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)
    cfg = get_cfg()
    sim_module = SimilarityModule(cfg, input_dim=D)
    out = sim_module(query, support, mask, size)
    print("Output:", out)
    assert torch.isfinite(out).all()
    print("✅ Passed")


def test_high_temperature():
    print("\n[TEST] Extreme Temperature")
    B, N, D = 2, 5, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)
    for temp in [0.01, 100.0]:
        cfg = get_cfg(temperature=temp)
        sim_module = SimilarityModule(cfg, input_dim=D)
        out = sim_module(query, support, mask, size)
        print(f"Temp={temp}: {out.squeeze().tolist()}")
        assert torch.isfinite(out).all()
    print("✅ Passed")


def test_all_positive_negative():
    print("\n[TEST] All Positive / All Negative")
    B, N, D = 2, 4, 8
    query = torch.ones(B, 1, D)
    support = torch.ones(B, N, D) * 2
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)
    cfg = get_cfg(posWeight=1.0, negWeight=2.0)
    sim_module = SimilarityModule(cfg, input_dim=D)
    out = sim_module(query, support, mask, size)
    print("All positive support output:", out)
    assert torch.isfinite(out).all()

    support = -torch.ones(B, N, D) * 2
    out = sim_module(query, support, mask, size)
    print("All negative support output:", out)
    assert torch.isfinite(out).all()
    print("✅ Passed")


def test_nan_inputs():
    print("\n[TEST] NaN Handling")
    B, N, D = 2, 4, 8
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    support[0, 1, 3] = float("nan")  # introduce NaN
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)
    cfg = get_cfg()
    sim_module = SimilarityModule(cfg, input_dim=D)
    out = sim_module(query, support, mask, size)
    print("Output with NaN in support:", out)
    assert torch.isfinite(out).all()
    print("✅ Passed")


def test_multihead_consistency():
    print("\n[TEST] Multi-head Consistency")
    B, N, D = 2, 5, 16
    query = torch.randn(B, 1, D)
    support = torch.randn(B, N, D)
    mask = torch.ones(B, N).bool()
    size = torch.full((B,), N)

    cfg_single = get_cfg(numHeads=1)
    sim_single = SimilarityModule(cfg_single, input_dim=D)
    out_single = sim_single(query, support, mask, size)

    cfg_multi = get_cfg(numHeads=4)
    sim_multi = SimilarityModule(cfg_multi, input_dim=D)
    out_multi = sim_multi(query, support, mask, size)

    print("Single-head output:", out_single.squeeze().tolist())
    print("Multi-head output:", out_multi.squeeze().tolist())
    assert torch.isfinite(out_multi).all()
    print("✅ Passed")


if __name__ == "__main__":
    torch.manual_seed(42)

    # Original tests
    test_shapes_and_heads()
    test_aggregation_methods()
    test_topk()
    test_temperature_scaling()
    test_pos_neg_weighting()
    test_variable_support_size_and_masking()
    test_gradient_flow()

    # Extended tests
    test_empty_support()
    test_single_support_vector()
    test_high_temperature()
    test_all_positive_negative()
    test_nan_inputs()
    test_multihead_consistency()

    print("\nAll tests passed ✅")


[TEST] Shapes & Heads
Output shape: torch.Size([4, 1])
✅ Passed

[TEST] Aggregation Methods
sum: [0.009579318575561047, 0.0003157868923153728]
softmax: [0.022361472249031067, 0.0030978741124272346]
logsumexp: [1.630190372467041, 1.6113086938858032]
✅ Passed

[TEST] Top-K Selection
Top-K=None: [0.00021123839542269707, -0.0006575907464139163]
Top-K=3: [0.015330187976360321, 0.015437143854796886]
Top-K=5: [0.008511207066476345, 0.008020715788006783]
✅ Passed

[TEST] Temperature Scaling
Temp=0.5: [0.004669899586588144, -0.002411243272945285]
Temp=1.0: [0.009339799173176289, -0.00482248654589057]
Temp=2.0: [0.018679598346352577, -0.00964497309178114]
✅ Passed

[TEST] Pos/Neg Weighting
pos=1.0, neg=1.0: [0.02988637238740921, -0.01140357181429863]
pos=2.0, neg=0.5: [0.08124704658985138, 0.027484312653541565]
pos=0.5, neg=2.0: [-0.006531114224344492, -0.05599324405193329]
✅ Passed

[TEST] Variable Support Size + Masking
Output: tensor([[-0.0066],
        [-0.0085],
        [-0.0067]])
✅ Passe

### Old Code

In [ ]:
import torch
from omegaconf import OmegaConf


def similarity_module(
    query_embedding: torch.Tensor,
    support_set_embeddings: torch.Tensor,
    padding_mask: torch.Tensor,
    support_set_size: torch.Tensor,
    cfg: OmegaConf,
) -> torch.Tensor:
    """
    Similarity Module

    Computes activity prediction for a query by aggregating similarities
    with support set embeddings.

    This module is applied twice:
    - active support set
    - inactive support set

    Args:
        query_embedding: [B, 1, D]
        support_set_embeddings: [B, N, D]
        padding_mask: [B, N] (True = valid, False = padding)
        support_set_size: [B]
        cfg: Hydra config

    Returns:
        similarity score: [B, 1]
    """

    # -------------------------------------------------
    # Optional L2 normalization (cosine similarity)
    # -------------------------------------------------
    if cfg.model.similarityModule.l2Norm:
        query_embedding = torch.nn.functional.normalize(
            query_embedding, dim=-1, eps=1e-8
        )
        support_set_embeddings = torch.nn.functional.normalize(
            support_set_embeddings, dim=-1, eps=1e-8
        )

    # -------------------------------------------------
    # Similarity computation
    # -------------------------------------------------
    similarities = torch.matmul(
        query_embedding,
        support_set_embeddings.transpose(1, 2)
    )  # [B, 1, N]

    # -------------------------------------------------
    # Mask padding
    # -------------------------------------------------
    mask = padding_mask.bool().unsqueeze(1)  # [B,1,N]

    similarities = torch.nan_to_num(similarities)
    similarities = similarities * mask

    # -------------------------------------------------
    # Aggregate similarities
    # -------------------------------------------------
    similarity_sums = similarities.sum(dim=2)  # [B,1]

    # -------------------------------------------------
    # Scaling
    # -------------------------------------------------
    stabilizer = 1e-8
    N = support_set_size.reshape(-1, 1).float()

    if cfg.model.similarityModule.scaling == "1/N":
        similarity_sums = similarity_sums / (2.0 * N + stabilizer)

    elif cfg.model.similarityModule.scaling == "1/sqrt(N)":
        similarity_sums = similarity_sums / (2.0 * torch.sqrt(N) + stabilizer)

    return similarity_sums


    # Double-check mask semantics
    # Add assertion checks
    # Add temperature scaling
    # Add optional softmax weighting (attention version)
    # Separate positive / negative importance
    # Replace sum with log-sum-exp
    # Top-k similarity
    # Interpret similarity as energy
    # multi-head similarity